# Семинар. Профилирование, оптимизация, векторизация и Numba

Ноутбук основан на материалах лекции по теме «Лекция 8: Оптимизация выполнения кода, векторизация, Numba»

## Цель работы

Научиться ускорять численные алгоритмы на Python с помощью CPU-инструментов Numba:

- `@njit` и режим `nopython`;
- `@vectorize` для скалярных ufunc;
- `@guvectorize` для generalized ufunc;
- `prange` и `parallel=True` для параллельных циклов;
- `fastmath` для контролируемого ослабления строгой IEEE-арифметики;
- проверка корректности и базовая оценка времени выполнения.

## Правила
1. В начале ноутбука задаётся `STUDENT_ID`. Все массивы, размеры задач, параметры формул и контрольные значения генерируются детерминированно из `STUDENT_ID`.
1. Не менять служебные функции генерации данных и проверки.
1. В каждой задаче использовать Numba, а не только NumPy-векторизацию.
1. После решения каждой задачи выполнить ячейку проверки.
1. Для задач с `parallel=True` не использовать запись в общую переменную без редукции или независимых индексов.

In [ ]:
# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"

In [ ]:
# 0. Настройка окружения и индивидуального варианта

import hashlib
import math
import time
import numpy as np
import numba
from numba import njit, vectorize, guvectorize, prange


def seed_from_student_id(student_id: str) -> int:
    text = str(student_id).strip()
    if text in {"", "REPLACE_ME", "PUT_YOUR_STUDENT_ID_HERE"} or len(text) < 3:
        raise ValueError("Сначала укажи реальный STUDENT_ID в ячейке настройки.")
    digest = hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "little") % (2**32)


SEED = seed_from_student_id(STUDENT_ID)
VARIANT = SEED % 997 + 3
print(f"STUDENT_ID={STUDENT_ID!r}; SEED={SEED}; VARIANT={VARIANT}")


def task_rng(task_no: int) -> np.random.Generator:
    return np.random.default_rng((SEED + task_no * 1009) % (2**32))


def assert_allclose(name: str, got, expected, rtol=1e-9, atol=1e-9):
    got_arr = np.asarray(got)
    exp_arr = np.asarray(expected)
    if got_arr.shape != exp_arr.shape:
        raise AssertionError(f"{name}: неверная форма результата: {got_arr.shape}, ожидалось {exp_arr.shape}")
    if not np.allclose(got_arr, exp_arr, rtol=rtol, atol=atol, equal_nan=True):
        diff = np.max(np.abs(got_arr - exp_arr))
        raise AssertionError(f"{name}: результат отличается от эталона; max abs diff = {diff}")
    print(f"{name}: корректно")


def assert_numba_dispatcher(name: str, fn):
    if not hasattr(fn, "signatures") or len(fn.signatures) == 0:
        raise AssertionError(f"{name}: функция должна быть скомпилирована Numba через @njit/@jit")
    print(f"{name}: Numba signatures = {fn.signatures}")


def assert_numba_ufunc(name: str, fn):
    if not hasattr(fn, "types") or len(fn.types) == 0:
        raise AssertionError(f"{name}: функция должна быть Numba ufunc/gufunc через @vectorize/@guvectorize")
    print(f"{name}: Numba ufunc types = {fn.types}")


def time_call(fn, *args, repeat: int = 3):
    best = float("inf")
    result = None
    for _ in range(repeat):
        start = time.perf_counter()
        result = fn(*args)
        elapsed = time.perf_counter() - start
        best = min(best, elapsed)
    return best, result

STUDENT_ID='123456'; SEED=3682203439; VARIANT=291


In [ ]:
# Служебные функции генерации данных и эталонные реализации
# Эти функции используются только для проверки корректности. Основное решение должно быть написано через Numba.


def task1_data():
    rng = task_rng(1)
    n = 1800 + VARIANT % 400
    m = 18 + VARIANT % 5
    X = rng.normal(size=(n, m)).astype(np.float64)
    w = rng.uniform(-1.5, 1.5, size=m).astype(np.float64)
    alpha = np.float64(0.05 + (VARIANT % 7) * 0.01)
    return X, w, alpha


def ref_row_energy(X, w, alpha):
    out = np.empty(X.shape[0], dtype=np.float64)
    for i in range(X.shape[0]):
        s = 0.0
        for j in range(X.shape[1]):
            v = X[i, j]
            s += w[j] * (v * v + alpha * math.sin(v))
        out[i] = s
    return out


def task2_data():
    rng = task_rng(2)
    n = 120_000 + int(VARIANT % 50_000)
    values = rng.integers(0, 10_000_000, size=n, dtype=np.int64)
    base = np.int64(7 + VARIANT % 10)
    digit = np.int64(VARIANT % int(base))
    return values, digit, base


def ref_count_digit(values, digit, base):
    total = 0
    for raw in values:
        v = abs(int(raw))
        if v == 0:
            if int(digit) == 0:
                total += 1
        while v > 0:
            v, d = divmod(v, int(base))
            if d == int(digit):
                total += 1
    return total


def task3_data():
    rng = task_rng(3)
    n = 160_000 + VARIANT % 40_000
    x = rng.normal(loc=0.1, scale=2.0, size=n).astype(np.float64)
    a = np.float64(0.3 + (VARIANT % 5) * 0.1)
    b = np.float64(1.1 + (VARIANT % 7) * 0.05)
    c = np.float64(0.5 + (VARIANT % 3) * 0.25)
    return x, a, b, c


def ref_element_score(x, a, b, c):
    out = np.empty_like(x, dtype=np.float64)
    neg = x < 0.0
    mid = (x >= 0.0) & (x < c)
    high = ~neg & ~mid
    out[neg] = -a * np.sqrt(np.abs(x[neg])) + np.sin(b * x[neg])
    out[mid] = b * np.log1p(x[mid] + c) + x[mid] * x[mid]
    out[high] = np.sqrt(x[high] + c) - a * np.cos(x[high])
    return out


def task4_data():
    rng = task_rng(4)
    n = 2500 + VARIANT % 600
    k = 16 + VARIANT % 8
    y_true = rng.normal(size=(n, k)).astype(np.float64)
    y_pred = y_true + rng.normal(scale=0.7, size=(n, k)).astype(np.float64)
    delta = np.float64(0.5 + (VARIANT % 5) * 0.15)
    return y_pred, y_true, delta


def ref_huber_rows(pred, true, delta):
    diff = pred - true
    absdiff = np.abs(diff)
    return np.where(absdiff <= delta, 0.5 * diff * diff, delta * (absdiff - 0.5 * delta)).sum(axis=1)


def task5_data():
    rng = task_rng(5)
    n = 420 + (VARIANT % 80)
    m = 350 + (VARIANT % 70)
    d = 8 + (VARIANT % 5)
    A = rng.normal(size=(n, d)).astype(np.float64)
    B = rng.normal(size=(m, d)).astype(np.float64)
    return A, B


def ref_pairwise_sqdist(A, B):
    return ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)


def task6_data():
    rng = task_rng(6)
    n = 2500 + VARIANT % 700
    nnz_per_row = 5 + VARIANT % 6
    indptr = np.zeros(n + 1, dtype=np.int64)
    indices = np.empty(n * nnz_per_row, dtype=np.int64)
    data = np.empty(n * nnz_per_row, dtype=np.float64)
    pos = 0
    for i in range(n):
        cols = np.sort(rng.choice(n, size=nnz_per_row, replace=False)).astype(np.int64)
        vals = rng.normal(size=nnz_per_row)
        indices[pos : pos + nnz_per_row] = cols
        data[pos : pos + nnz_per_row] = vals
        indptr[i] = pos
        pos += nnz_per_row
    indptr[n] = pos
    x = rng.normal(size=n).astype(np.float64)
    return data, indices, indptr, x, n


def ref_csr_matvec(data, indices, indptr, x, n_rows):
    y = np.empty(n_rows, dtype=np.float64)
    for i in range(n_rows):
        s = 0.0
        for pos in range(indptr[i], indptr[i + 1]):
            s += data[pos] * x[indices[pos]]
        y[i] = s
    return y


SMOOTH_RADIUS = 2 + (VARIANT % 2)


def task7_data():
    rng = task_rng(7)
    batch = 1200 + VARIANT % 300
    n = 96 + VARIANT % 32
    return rng.normal(size=(batch, n)).astype(np.float64)


def ref_edge_smooth(x):
    out = np.empty_like(x)
    r = SMOOTH_RADIUS
    for row in range(x.shape[0]):
        for i in range(x.shape[1]):
            s = 0.0
            wsum = 0.0
            for off in range(-r, r + 1):
                j = i + off
                if j < 0:
                    j = 0
                if j >= x.shape[1]:
                    j = x.shape[1] - 1
                w = 1.0 / (1.0 + abs(off))
                s += w * x[row, j]
                wsum += w
            out[row, i] = s / wsum
    return out


def task8_data():
    rng = task_rng(8)
    n = 110 + VARIANT % 35
    a = rng.normal(size=n).astype(np.float64)
    b = (a + rng.normal(scale=0.5, size=n)).astype(np.float64)
    band = np.int64(8 + VARIANT % 8)
    return a, b, band


def ref_dtw_banded(a, b, band):
    n = len(a)
    m = len(b)
    inf = float("inf")
    dp = np.full((n + 1, m + 1), inf, dtype=np.float64)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        j_start = max(1, i - int(band))
        j_end = min(m, i + int(band)) + 1
        for j in range(j_start, j_end):
            cost = abs(float(a[i - 1]) - float(b[j - 1]))
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return dp[n, m]


def task9_data():
    rng = task_rng(9)
    h = 420 + VARIANT % 80
    w = 430 + VARIANT % 70
    img = rng.normal(size=(h, w)).astype(np.float64)
    lam = np.float64(0.12 + (VARIANT % 5) * 0.02)
    gamma = np.float64(0.01 + (VARIANT % 7) * 0.002)
    return img, lam, gamma


def ref_diffusion_step(img, lam, gamma):
    out = img.copy()
    gx = img[1:-1, 2:] - img[1:-1, :-2]
    gy = img[2:, 1:-1] - img[:-2, 1:-1]
    lap = img[:-2, 1:-1] + img[2:, 1:-1] + img[1:-1, :-2] + img[1:-1, 2:] - 4 * img[1:-1, 1:-1]
    out[1:-1, 1:-1] = img[1:-1, 1:-1] + lam * lap - gamma * (gx * gx + gy * gy)
    return out


def task10_params():
    n_walks = 20_000 + VARIANT % 5000
    n_steps = 80 + VARIANT % 30
    boundary = 18 + VARIANT % 7
    seed = np.uint64(SEED ^ 0x9E3779B97F4A7C15)
    return np.int64(n_walks), np.int64(n_steps), np.int64(boundary), seed


def ref_random_walk_stats(n_walks, n_steps, boundary, seed):
    mult = 6364136223846793005
    inc = 1442695040888963407
    mask = (1 << 64) - 1
    hits = 0
    abs_sum = 0.0
    for walk in range(int(n_walks)):
        state = (int(seed) + walk * 2 + 1) & mask
        x = 0
        y = 0
        hit = False
        for _ in range(int(n_steps)):
            state = (state * mult + inc) & mask
            step = (state >> 62) & 3
            if step == 0:
                x += 1
            elif step == 1:
                x -= 1
            elif step == 2:
                y += 1
            else:
                y -= 1
            if abs(x) + abs(y) >= int(boundary):
                hit = True
        if hit:
            hits += 1
        abs_sum += abs(x) + abs(y)
    return hits, abs_sum / float(n_walks)

## Задание 1. `@njit`: взвешенная энергия строк матрицы


Реализуй функцию `row_energy_numba(X, w, alpha)`, которая для каждой строки матрицы `X` вычисляет

$$E_i = \sum_{j=1}^{m} w_j \cdot \left(x_{ij}^2 + \alpha \sin(x_{ij})\right)$$

Требования:

- использовать `@njit`;
- не использовать в теле функции `np.sum`, broadcasting и списковые включения;
- результат — массив `float64` длины `X.shape[0]`.

In [ ]:
@njit
def row_energy_numba(X, w, alpha):
    # ВАШ КОД

In [ ]:
X, w, alpha = task1_data()
got = row_energy_numba(X, w, alpha)
expected = ref_row_energy(X, w, alpha)
assert_numba_dispatcher("Задание 1", row_energy_numba)
assert_allclose("Задание 1", got, expected)

Задание 1: Numba signatures = [(Array(float64, 2, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), float64)]
Задание 1: корректно


## Задание 2. `@njit`: подсчет цифры в индивидуальной системе счисления


Дан массив неотрицательных целых чисел `values`. Для индивидуальных `base` и `digit`, зависящих от `STUDENT_ID`, посчитай, сколько раз цифра `digit` встречается в записи всех чисел массива в системе счисления `base`.

Требования:

- использовать `@njit`;
- не переводить числа в строки;
- обработать случай `value == 0`;
- вернуть один `int`.

In [ ]:
@njit
def count_digit_numba(values, digit, base):
    # ВАШ КОД

In [ ]:
values, digit, base = task2_data()
got = count_digit_numba(values, digit, base)
expected = ref_count_digit(values, digit, base)
assert_numba_dispatcher("Задание 2", count_digit_numba)
if int(got) != int(expected):
    raise AssertionError(f"Задание 2: got={got}, expected={expected}, digit={digit}, base={base}")
print("Задание 2: корректно")
print("digit/base:", int(digit), int(base))

Задание 2: Numba signatures = [(Array(int64, 1, 'C', False, aligned=True), int64, int64)]
Задание 2: корректно
digit/base: 3 8


## Задание 3. `@vectorize`: собственная скалярная ufunc


Реализуй Numba-ufunc `element_score_numba(x, a, b, c)` для поэлементного преобразования:

$f(x)= \begin{cases} -a\sqrt{|x|}+\sin(bx), & x < 0, \\ b\log(1+x+c)+x^2, & 0 \le x < c, \\ \sqrt{x+c}-a\cos(x), & x \ge c. \end{cases}$

Требования:

- использовать `@vectorize(..., nopython=True)`;
- функция должна принимать скаляры и работать по массиву как ufunc;
- использовать функции из `math`, а не NumPy внутри ядра.

In [ ]:
@vectorize(["float64(float64, float64, float64, float64)"], nopython=True)
def element_score_numba(x, a, b, c):
    # ВАШ КОД

In [ ]:
x, a, b, c = task3_data()
got = element_score_numba(x, a, b, c)
expected = ref_element_score(x, a, b, c)
assert_numba_ufunc("Задание 3", element_score_numba)
assert_allclose("Задание 3", got, expected)
print("a,b,c:", float(a), float(b), float(c))

Задание 3: Numba ufunc types = ['dddd->d']
Задание 3: корректно
a,b,c: 0.4 1.3 0.5


## Задание 4. `@guvectorize`: Huber loss по строкам


Реализуй generalized ufunc `huber_rows_numba(pred, true, delta)`, которая для каждой пары строк `pred[i]` и `true[i]` возвращает сумму Huber loss:

$L_\delta(r)= \begin{cases} 0.5r^2, & |r| \le \delta,\\ \delta(|r|-0.5\delta), & |r| > \delta. \end{cases}$

Требования:

- использовать `@guvectorize` с сигнатурой размерностей `'(n),(n),()->()'`;
- не возвращать значение через `return`; результат записывать в `out[0]`;
- результат для входа формы `(batch, n)` должен иметь форму `(batch,)`.

In [ ]:
@guvectorize(["void(float64[:], float64[:], float64, float64[:])"], "(n),(n),()->()", nopython=True)
def huber_rows_numba(pred, true, delta, out):
    # ВАШ КОД

In [ ]:
pred, true, delta = task4_data()
got = huber_rows_numba(pred, true, delta)
expected = ref_huber_rows(pred, true, delta)
assert_numba_ufunc("Задание 4", huber_rows_numba)
assert_allclose("Задание 4", got, expected)
print("delta:", float(delta))

Задание 4: Numba ufunc types = ['ddd->d']
Задание 4: корректно
delta: 0.65


## Задание 5. `@njit(parallel=True)`: попарные квадраты расстояний


Реализуй `pairwise_sqdist_numba(A, B)`, которая возвращает матрицу `D`, где

$D_{ij}=\sum_k (A_{ik}-B_{jk})^2$

Требования:

- использовать `@njit(parallel=True)` и `prange` по независимому внешнему циклу;
- не использовать broadcasting `A[:, None, :] - B[None, :, :]` в решении;
- результат — матрица формы `(A.shape[0], B.shape[0])`.

In [ ]:
@njit(parallel=True)
def pairwise_sqdist_numba(A, B):
    # ВАШ КОД

In [ ]:
A, B = task5_data()
got = pairwise_sqdist_numba(A, B)
expected = ref_pairwise_sqdist(A, B)
assert_numba_dispatcher("Задание 5", pairwise_sqdist_numba)
assert_allclose("Задание 5", got, expected)
print("shape:", got.shape)

Задание 5: Numba signatures = [(Array(float64, 2, 'C', False, aligned=True), Array(float64, 2, 'C', False, aligned=True))]
Задание 5: корректно
shape: (471, 361)


## Задание 6. `@njit(parallel=True)`: умножение CSR-матрицы на вектор


Реализуй `csr_matvec_numba(data, indices, indptr, x, n_rows)` для разреженной матрицы в формате CSR.

CSR-формат:

- `data[pos]` — ненулевое значение;
- `indices[pos]` — номер столбца;
- `indptr[i] : indptr[i + 1]` — диапазон элементов строки `i`.

Требования:

- использовать `@njit(parallel=True)`;
- внешний цикл по строкам распараллелить через `prange`;
- не собирать плотную матрицу.

In [ ]:
@njit(parallel=True)
def csr_matvec_numba(data, indices, indptr, x, n_rows):
    # ВАШ КОД

In [ ]:
data, indices, indptr, x, n_rows = task6_data()
got = csr_matvec_numba(data, indices, indptr, x, n_rows)
expected = ref_csr_matvec(data, indices, indptr, x, n_rows)
assert_numba_dispatcher("Задание 6", csr_matvec_numba)
assert_allclose("Задание 6", got, expected)
print("n_rows, nnz:", n_rows, len(data))

Задание 6: Numba signatures = [(Array(float64, 1, 'C', False, aligned=True), Array(int64, 1, 'C', False, aligned=True), Array(int64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), int64)]
Задание 6: корректно
n_rows, nnz: 2791 22328


## Задание 7. `@guvectorize`: сглаживание одномерных сигналов


Реализуй generalized ufunc `edge_smooth_numba(x)`, которая сглаживает один вектор с радиусом `SMOOTH_RADIUS`. Для позиции `i` нужно взять окно `i-r ... i+r`; выход за границу заменяется ближайшим допустимым индексом. Вес смещения `off` равен:

\[
w(off)=\frac{1}{1+|off|}.
\]

Требования:

- использовать `@guvectorize` с сигнатурой `'(n)->(n)'`;
- вход формы `(batch, n)` должен обрабатываться построчно;
- результат записывать в выходной массив `out`.

In [ ]:
@guvectorize(["void(float64[:], float64[:])"], "(n)->(n)", nopython=True)
def edge_smooth_numba(x, out):
    # ВАШ КОД

In [ ]:
X = task7_data()
got = edge_smooth_numba(X)
expected = ref_edge_smooth(X)
assert_numba_ufunc("Задание 7", edge_smooth_numba)
assert_allclose("Задание 7", got, expected)
print("SMOOTH_RADIUS:", SMOOTH_RADIUS)

Задание 7: Numba ufunc types = ['d->d']
Задание 7: корректно
SMOOTH_RADIUS: 3


## Задание 8. `@njit`: banded DTW для двух временных рядов


Реализуй `dtw_banded_numba(a, b, band)` — расстояние Dynamic Time Warping с ограничением полосы `|i-j| <= band`.

Требования:

- использовать `@njit`;
- не хранить всю матрицу `n x m`, достаточно двух строк DP;
- стоимость сопоставления элементов — `abs(a[i] - b[j])`;
- вернуть одно число `float64`.

In [ ]:
@njit
def dtw_banded_numba(a, b, band):
    # ВАШ КОД

In [ ]:
a, b, band = task8_data()
got = dtw_banded_numba(a, b, band)
expected = ref_dtw_banded(a, b, band)
assert_numba_dispatcher("Задание 8", dtw_banded_numba)
if not math.isclose(float(got), float(expected), rel_tol=1e-9, abs_tol=1e-9):
    raise AssertionError(f"Задание 8: got={got}, expected={expected}")
print("Задание 8: корректно")
print("band:", int(band))

Задание 8: Numba signatures = [(Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), int64)]
Задание 8: корректно
band: 11


## Задание 9. `@njit(parallel=True, fastmath=True)`: один шаг stencil-фильтра


Реализуй `diffusion_step_numba(img, lam, gamma)` для двумерного массива. Границы оставить без изменений. Для внутренних пикселей:

$$
\text{out}_{ij}=x_{ij}+\lambda(x_{i-1,j}+x_{i+1,j}+x_{i,j-1}+x_{i,j+1}-4x_{ij})-
\gamma\left((x_{i,j+1}-x_{i,j-1})^2+(x_{i+1,j}-x_{i-1,j})^2\right)
$$

Требования:

- использовать `@njit(parallel=True, fastmath=True)`;
- внешний цикл по строкам — `prange`;
- не изменять входной массив `img` на месте.

In [ ]:
@njit(parallel=True, fastmath=True)
def diffusion_step_numba(img, lam, gamma):
    # ВАШ КОД

In [ ]:
img, lam, gamma = task9_data()
got = diffusion_step_numba(img, lam, gamma)
expected = ref_diffusion_step(img, lam, gamma)
assert_numba_dispatcher("Задание 9", diffusion_step_numba)
assert_allclose("Задание 9", got, expected, rtol=1e-8, atol=1e-8)
print("lam,gamma:", float(lam), float(gamma))

Задание 9: Numba signatures = [(Array(float64, 2, 'C', False, aligned=True), float64, float64)]
Задание 9: корректно
lam,gamma: 0.13999999999999999 0.018000000000000002


## Задание 10. `@njit`: детерминированное моделирование случайных блужданий


Реализуй `random_walk_stats_numba(n_walks, n_steps, boundary, seed)`. Нужно смоделировать `n_walks` двумерных случайных блужданий длины `n_steps`, используя заданный линейный конгруэнтный генератор. На каждом шаге старшие 2 бита состояния задают направление: `0` — вправо, `1` — влево, `2` — вверх, `3` — вниз.

Вернуть пару:

1. сколько траекторий хотя бы раз достигли границы `abs(x)+abs(y) >= boundary`;
2. среднее финальное значение `abs(x)+abs(y)`.

Требования:

- использовать `@njit`;
- не использовать `np.random` внутри JIT-функции;
- состояние генератора обновлять в арифметике `uint64`.

In [ ]:
@njit
def random_walk_stats_numba(n_walks, n_steps, boundary, seed):
    # ВАШ КОД

In [ ]:
n_walks, n_steps, boundary, seed = task10_params()
got = random_walk_stats_numba(n_walks, n_steps, boundary, seed)
expected = ref_random_walk_stats(n_walks, n_steps, boundary, seed)
assert_numba_dispatcher("Задание 10", random_walk_stats_numba)
if int(got[0]) != int(expected[0]) or not math.isclose(float(got[1]), float(expected[1]), rel_tol=1e-12, abs_tol=1e-12):
    raise AssertionError(f"Задание 10: got={got}, expected={expected}")
print("Задание 10: корректно")
print("n_walks,n_steps,boundary:", int(n_walks), int(n_steps), int(boundary))

Задание 10: Numba signatures = [(int64, int64, int64, uint64)]
Задание 10: корректно
n_walks,n_steps,boundary: 20291 101 22


## Дополнительная ячейка: базовая оценка времени

Эта ячейка не заменяет полноценное профилирование, но помогает увидеть эффект JIT-компиляции. Первый вызов Numba-функции включает компиляцию, поэтому в замерах используется повторный вызов после проверок.

In [ ]:
# Необязательная быстрая сводка времени выполнения.
# В разных средах абсолютное время будет отличаться, поэтому это диагностический, а не оценочный блок.
benchmarks = []

X, w, alpha = task1_data()
t, _ = time_call(row_energy_numba, X, w, alpha)
benchmarks.append(("1 row_energy_numba", t))

values, digit, base = task2_data()
t, _ = time_call(count_digit_numba, values, digit, base)
benchmarks.append(("2 count_digit_numba", t))

x, a, b, c = task3_data()
t, _ = time_call(element_score_numba, x, a, b, c)
benchmarks.append(("3 element_score_numba", t))

pred, true, delta = task4_data()
t, _ = time_call(huber_rows_numba, pred, true, delta)
benchmarks.append(("4 huber_rows_numba", t))

A, B = task5_data()
t, _ = time_call(pairwise_sqdist_numba, A, B)
benchmarks.append(("5 pairwise_sqdist_numba", t))

data, indices, indptr, x, n_rows = task6_data()
t, _ = time_call(csr_matvec_numba, data, indices, indptr, x, n_rows)
benchmarks.append(("6 csr_matvec_numba", t))

X = task7_data()
t, _ = time_call(edge_smooth_numba, X)
benchmarks.append(("7 edge_smooth_numba", t))

a, b, band = task8_data()
t, _ = time_call(dtw_banded_numba, a, b, band)
benchmarks.append(("8 dtw_banded_numba", t))

img, lam, gamma = task9_data()
t, _ = time_call(diffusion_step_numba, img, lam, gamma)
benchmarks.append(("9 diffusion_step_numba", t))

n_walks, n_steps, boundary, seed = task10_params()
t, _ = time_call(random_walk_stats_numba, n_walks, n_steps, boundary, seed)
benchmarks.append(("10 random_walk_stats_numba", t))

for name, seconds in benchmarks:
    print(f"{name:32s}: {seconds:.6f} s")

1 row_energy_numba              : 0.000849 s
2 count_digit_numba             : 0.010018 s
3 element_score_numba           : 0.005905 s
4 huber_rows_numba              : 0.000115 s
5 pairwise_sqdist_numba         : 0.001752 s
6 csr_matvec_numba              : 0.000075 s
7 edge_smooth_numba             : 0.001589 s
8 dtw_banded_numba              : 0.000034 s
9 diffusion_step_numba          : 0.000314 s
10 random_walk_stats_numba      : 0.023592 s
